<a href="https://colab.research.google.com/github/Joseonggwan/medical-data-analysis-project/blob/main/medi_analysis_final_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#의료비 예측 및 고위험 환자 분석 프로젝트 보고서
20252416 조성관

##1.프로젝트 개요
프로젝트 배경

의료비는 개인의 건강 상태, 생활 습관, 연령, 흡연 여부 등 다양한 요인의 영향을 받는다.
특히 최근 의료비 증가 문제는 보험 산업 및 의료 정책 분야에서 중요한 이슈로 다루어지고 있으며, 고위험 환자를 조기에 식별하는 기술의 필요성이 커지고 있다.

본 프로젝트에서는 보험 가입자 데이터를 기반으로 머신러닝 기법을 활용하여 개인 의료비를 예측하고, 의료비 증가에 영향을 미치는 핵심 요인을 분석하고자 하였다.

또한 단순 예측에 그치지 않고:

######-고위험 환자 그룹 탐색
######-설명 가능한 AI(SHAP) 기반 해석
######-고비용 환자 분류

등을 함께 수행하여 실제 의료 데이터 분석 흐름에 가까운 프로젝트를 구현하는 것을 목표로 하였다.

##2.데이터셋 소개
데이터 출처: Medical Cost Personal Dataset (Kaggle)
######데이터 수: 총 1,338명
######주요 변수:
######age: 나이
######sex: 성별
######bmi: 체질량지수(BMI)
######children: 자녀 수
######smoker: 흡연 여부
######region: 거주 지역
######charges: 의료비(예측 대상)

### 데이터 특징
######의료비(charges)는 강한 우측 편향(right-skewed) 분포를 보였으며, 일부 고액 의료비 환자가 존재하였다.

######또한 흡연 여부에 따라 의료비 분포 차이가 매우 크게 나타났으며, BMI 및 나이와 같은 변수와의 상호작용 가능성도 확인할 수 있었다.

##3. EDA 및 데이터 분석
######(1) 의료비 분포 분석

의료비 데이터는 낮은 의료비 구간에 데이터가 집중되어 있었으며, 일부 고액 환자들로 인해 긴 우측 꼬리를 가지는 형태를 보였다.

이러한 구조는 선형 모델 학습 안정성을 저하시킬 수 있기 때문에 로그 변환(log1p)을 적용하여 분포를 안정화하였다.

로그 변환 이후 왜도(skewness)는 1.52에서 -0.09 수준으로 감소하였으며, 분포가 보다 대칭적인 형태로 개선되었다.

또한 이상치 제거 대신 로그 변환을 선택하였다.
그 이유는 고액 의료비 환자가 단순 오류 데이터가 아니라 실제 고위험 환자일 가능성이 높다고 판단하였기 때문이다.

(2) 주요 변수 분석

Violin Plot과 Box Plot을 활용하여 변수별 의료비 분포를 분석하였다.

성별(sex)

남성이 여성보다 평균 의료비가 약간 높게 나타났으나, 전체 분포 형태는 유사하였다.

따라서 성별 변수의 영향력은 제한적인 것으로 판단하였다.

흡연 여부(smoker)

흡연자는 비흡연자보다 평균 의료비가 약 4배 이상 높게 나타났다.

또한 단순 평균 차이뿐 아니라 전체 의료비 분포 자체가 크게 달랐으며, 고액 의료비 환자가 집중적으로 나타났다.

이를 통해 흡연 여부가 의료비 예측에서 가장 핵심적인 변수임을 확인할 수 있었다.

지역(region)

지역별 평균 의료비 차이는 일부 존재하였으나, 흡연 여부에 비해서는 영향력이 상대적으로 작게 나타났다.

##4.Feature Engineering

EDA 결과를 기반으로 의료비 증가에 영향을 줄 수 있는 상호작용(interaction) 기반 파생변수를 생성하였다.

생성한 주요 변수는 다음과 같다.

######obese: BMI ≥ 30 여부
######smoker_bin: 흡연 여부 이진화
######smoker_obese: 흡연 × 비만
######age_smoker: 나이 × 흡연
######bmi_smoker: BMI × 흡연
######has_children: 자녀 유무

특히 흡연과 BMI, 나이의 결합 효과가 의료비 증가에 중요한 영향을 줄 것으로 판단하여 interaction 변수를 중점적으로 생성하였다.

##5. 데이터 전처리

모델 학습을 위해 다음과 같은 전처리를 수행하였다.

###Train/Test Split

데이터를 8:2 비율로 학습 데이터와 테스트 데이터로 분리하였다.

또한 데이터 누수(Data Leakage)를 방지하기 위해 Train 데이터에 대해서만 fit을 수행하고 Test 데이터에는 transform만 적용하였다.

###수치형 변수 처리

StandardScaler를 적용하여 평균 0, 표준편차 1 기준으로 정규화하였다.

###범주형 변수 처리

OneHotEncoder를 사용하여 문자열 형태의 범주형 변수를 머신러닝 모델이 처리 가능한 숫자 형태로 변환하였다.

##6. 머신러닝 회귀 모델 비교

다양한 회귀 모델을 비교하여 의료비 예측 성능을 평가하였다.

###사용 모델:

Linear Regression
Ridge
Lasso
Random Forest
Gradient Boosting

###평가 지표:

R²
RMSE
MAE
MAPE
###결과

Random Forest 모델이 가장 높은 성능을 보였다.

R² ≈ 0.864
RMSE ≈ 4,599
MAE ≈ 2,557

또한 Gradient Boosting 역시 높은 성능을 보였으며, 선형 모델도 로그 변환과 Feature Engineering 적용 이후 성능이 크게 향상되었다.

반면 Lasso는 과도한 규제로 인해 성능이 크게 저하되었다.

##7. 교차검증(Cross Validation)

단순 Train/Test 평가만으로는 특정 데이터 분할에 과적합되었을 가능성을 완전히 배제하기 어렵다.

따라서 5-Fold Cross Validation을 수행하여 모델의 일반화 성능과 안정성을 검증하였다.

분석 결과 Random Forest와 Gradient Boosting은 높은 평균 R²와 낮은 표준편차를 동시에 보이며 안정적인 성능을 나타냈다.

##8. 하이퍼파라미터 튜닝

RandomizedSearchCV를 활용하여:

Random Forest
Gradient Boosting

모델의 최적 파라미터를 탐색하였다.

튜닝 이후 Gradient Boosting 모델이 가장 높은 성능을 보였다.

최종 R² ≈ 0.877

이를 통해 하이퍼파라미터 최적화가 모델 성능 향상에 효과적임을 확인하였다.

##9. SHAP 기반 설명가능 AI 분석

모델의 예측 원인을 해석하기 위해 SHAP(SHapley Additive exPlanations)을 활용하였다.

###주요 결과

가장 영향력이 큰 변수는:

######bmi_smoker
######age
######age_smoker
######smoker_obese

순으로 나타났다.

특히 bmi_smoker 변수의 영향력이 매우 크게 나타났는데, 이는 단순 BMI 자체보다 “흡연 상태에서 BMI가 증가하는 경우”가 의료비 증가에 훨씬 더 큰 영향을 준다는 것을 의미한다.

또한 SHAP Waterfall Plot을 통해 개별 환자의 의료비 예측 원인을 설명할 수 있었다.

##10. 클러스터링 분석

K-Means Clustering을 활용하여 환자 그룹을 자동 분류하였다.

Elbow Method와 Silhouette Score를 기반으로 최적 클러스터 수(K=2)를 선택하였다.

###클러스터 특징

####고위험군
######평균 의료비: 약 $32,322
######흡연율: 100%
######비만율: 약 54%

####중위험군
######평균 의료비: 약 $8,454
######흡연율: 약 0.4%

이를 통해 흡연 여부가 환자 위험군 분리에 핵심적인 역할을 한다는 점을 확인할 수 있었다.

##11. 고비용 환자 분류 모델

의료비 상위 25% 환자를 “고비용 환자”로 정의하고 분류 모델을 구축하였다.

###사용 모델:

######Logistic Regression
######Random Forest Classifier
######Gradient Boosting Classifier
###결과

Gradient Boosting 분류 모델이 가장 우수한 성능을 보였다.

######ROC-AUC ≈ 0.855
######AP ≈ 0.829
######F1-score ≈ 0.770

ROC Curve 및 Precision-Recall Curve 분석 결과에서도 안정적인 성능을 확인할 수 있었다.

##12. 결론

본 프로젝트에서는 머신러닝 기반 의료비 예측 및 고위험 환자 분석을 수행하였다.

분석 결과:

흡연 여부가 의료비 증가의 핵심 요인으로 나타났으며
특히 BMI 및 나이와 결합될 경우 의료비가 크게 증가하는 패턴을 확인할 수 있었다.

또한:

######Feature Engineering
######하이퍼파라미터 튜닝
######SHAP 기반 설명가능 AI
######클러스터링 및 분류 모델

등을 함께 적용함으로써 단순 예측을 넘어 의료비 증가 원인과 고위험 환자 특성까지 함께 분석할 수 있었다.

향후에는:

######더 큰 규모의 의료 데이터 적용
######실제 임상 데이터 활용
######Causal Inference 기반 정책 분석
######딥러닝 기반 모델 적용

등으로 확장 가능할 것으로 기대된다.